In [42]:
import pickle
import networkx as nx
from typing import Optional
import numpy as np
from torch_geometric.data import HeteroData
import torch
from sklearn.preprocessing import StandardScaler

In [2]:
def load_graph(filepath):
    with open(filepath, 'rb') as f:
        G = pickle.load(f)
    return G

In [43]:
CAUSAL_SIGN = {
    "increases":            +1,
    "directly_increases":   +1,
    "positive_correlation": +1,
    "decreases":            -1,
    "directly_decreases":   -1,
    "negative_correlation": -1,
}

def _edge_sign(rel: str) -> Optional[int]:
    rel = rel.lower()
    if rel.startswith("rev_"):
        rel = rel[4:]
    return CAUSAL_SIGN.get(rel)

def _build_directed_graph(G) -> nx.DiGraph:
    DG = nx.DiGraph()
    for src, tgt, rel, _ in G.edges(keys=True, data=True):
        if rel.lower().startswith("rev_"):
            continue
        DG.add_edge(src, tgt, rel=rel)
    return DG
    
def compute_topological_features(G, nodes: Optional[list] = None):
    """
    Compute topological features for specified nodes in the graph.
    Args:
        G: NetworkX graph object
        nodes: List of nodes to compute features for (defaults to all nodes)
    Returns:
        dict: Dictionary containing topological features for each node in `nodes`
    """
    print("Computing topological features...")

    if nodes is None:
        nodes = list(G.nodes())

    # 1. Basic features mapped to the exact order of `nodes`
    degrees = np.array([G.degree(n) for n in nodes])
    # 2. Average edge weight for the specified nodes
    avg_edge_weight = []
    for n in nodes:
        if G.degree(n) > 0:
            weights = [G.get_edge_data(n, m, default={}).get('weight', 1.0) for m in G.neighbors(n)]
            avg_edge_weight.append(np.mean(weights) if weights else 0.0)
        else:
            avg_edge_weight.append(0.0)
    avg_edge_weight = np.array(avg_edge_weight)

    # Convert MultiGraph to DiGraph for the following features
    DG = _build_directed_graph(G=G)
    # 1. clustering coefficient
    clustering = np.array([nx.clustering(DG, n) for n in nodes])

    # 2. Centrality measures (computed globally, then mapped to `nodes`)
    # Note: eigenvector_centrality can fail to converge if the graph is disconnected; max_iter is helpful.
    try:
        ev_cent_dict = nx.eigenvector_centrality(DG, weight='weight', max_iter=1000)
    except nx.PowerIterationFailedConvergence:
        ev_cent_dict = nx.eigenvector_centrality_numpy(DG, weight='weight')
        
    deg_cent_dict = nx.degree_centrality(DG)

    eigenvector_centrality = np.array([ev_cent_dict[n] for n in nodes])
    degree_centrality = np.array([deg_cent_dict[n] for n in nodes])

    # 3. neighborhood density
    neighborhood_densities = []
    for n in nodes:
        neighbors = list(DG.neighbors(n))
        k = len(neighbors)
        
        # If a node has fewer than 2 neighbors, density is technically 0 (no triangles/pairs possible)
        if k < 2:
            neighborhood_densities.append(0.0)
            continue
            
        # Maximum possible edges among k neighbors: for an undirected graph: k * (k - 1) / 2
        max_possible_edges = k * (k - 1) / 2
        
        # Induced subgraph of the neighbors
        subgraph = DG.subgraph(neighbors)
        actual_edges = subgraph.number_of_edges()
        
        # Calculate density
        density = actual_edges / max_possible_edges
        neighborhood_densities.append(density)
    neighborhood_densities = np.array(neighborhood_densities)

    return {
        # 'nodes': nodes,
        'degrees': degrees,
        'clustering': clustering,
        'eigenvector_centrality': eigenvector_centrality,
        'degree_centrality': degree_centrality,
        #'avg_edge_weight': avg_edge_weight,
        'neighborhood_density': neighborhood_densities
    }

In [52]:
G = load_graph('../../GateEmbeddingTask/HRGNN/results/ANMerge/AD_KG/ecdf_5/G_ANMerge_dual_hybrid_ecdf_5.pkl')

In [31]:
patient_nodes = []
node_mappings = {}
for node, attr in G.nodes(data=True):
    #print(attr)
    nt = attr.get('label',"") or attr.get('type', '')
    if nt == 'Patient':
        patient_nodes.append(node)
    if nt not in node_mappings:
        node_mappings[nt] = {}
    else:
        node_mappings[nt][node] = len(node_mappings[nt])
len(patient_nodes)

691

In [25]:
topo_features = compute_topological_features(G, patient_nodes)
topo_features

Computing topological features...


{'degrees': array([ 64,  83,  62,  66, 149,  62,  88, 182,  57, 188, 289,  83,  76,
         47,  50, 104,  97,  79,  69,  50, 265, 125, 175, 108,  67, 120,
        105, 110, 133,  95, 127,  56,  50, 273, 331, 158, 461,  44,  72,
        186,  75,  56, 185, 127,  54,  85, 173, 146, 168,  91,  95,  59,
         74,  67, 103,  90,  72,  82, 112, 148, 206, 477,  42,  88, 146,
         76, 229,  50, 118,  86,  55,  98, 143, 192, 272, 176, 132,  98,
        233, 344,  51,  58,  88, 217,  95,  69,  51,  64, 151, 158, 288,
        245, 181,  59, 411, 284,  89,  88, 166, 120, 119,  91, 112,  41,
        105,  66, 285, 137,  60,  63, 184, 161, 208,  88,  47,  70, 133,
        151, 221, 335, 223, 108, 133, 212, 305,  84,  75, 341,  79,  72,
        112, 125,  85,  77,  84, 110,  96, 221, 152,  79, 128, 161,  59,
        190, 320,  77, 140, 164, 125, 139, 126, 170,  90,  54,  57, 124,
         60,  83,  80, 110, 127, 114, 105, 206, 169,  66, 119, 139, 236,
         79,  71,  94,  94,  94, 115, 10

In [81]:
def convert_to_hetero_data(G: nx.MultiDiGraph):
    """
    Converts the hybrid Patient-Protein network into a PyTorch Geometric HeteroData object.
    """
    print("Starting conversion from NetworkX to HeteroData...")
    data = HeteroData()
    node_mappings = {} # {node_type: {node_name: integer_index}}
    
    # 1. Categorize Nodes and Map Indices
    for node, attrs in G.nodes(data=True):
        n_type = attrs.get('type') or attrs.get('label')
        
        if n_type not in node_mappings:
            node_mappings[n_type] = {}
        
        if node not in node_mappings[n_type]:
            node_mappings[n_type][node] = len(node_mappings[n_type])

    # 2. Process Patient Data
    p_map = node_mappings.get('Patient', {})
    if not p_map:
        raise ValueError("No 'Patient' nodes found in the graph to use as a feature dimension reference.")
    
    p_ids = sorted(p_map, key=p_map.get)
    patient_x = [G.nodes[pid]['x'] for pid in p_ids]
    data['Patient'].y = torch.tensor([G.nodes[pid]['y'] for pid in p_ids], dtype=torch.long)
        
    # Standardize node features
    scaler_x  = StandardScaler()
    patient_x = np.asarray(patient_x, dtype=np.float32)
    x         = scaler_x.fit_transform(patient_x)
    data['Patient'].x = torch.tensor(x, dtype=torch.float)
      
    # Compute topological features from Graph
    topological_features = compute_topological_features(G=G, nodes=p_ids)
    # Stack and standardize topological features
    feature_names = ['degrees', 'clustering', 'eigenvector_centrality', 'degree_centrality', 'neighborhood_density']
    
    topo_features = np.vstack([topological_features[name] for name in feature_names]).T
    scaler_topo = StandardScaler()
    topo_features = scaler_topo.fit_transform(topo_features)
    data['Patient'].topo_features=torch.tensor(topo_features, dtype=torch.float)
    data['Patient'].topo_scaler_mean=torch.tensor(scaler_topo.mean_, dtype=torch.float)
    data['Patient'].topo_scaler_scale=torch.tensor(scaler_topo.scale_, dtype=torch.float)

    # 3. Process ALL Node Types
    for n_type, mapping in node_mappings.items():    
        num_nodes = len(mapping)
        data[n_type].num_nodes = num_nodes
        
    # 4. Process Edges with separation
    static_edges = {}
    dynamic_edges = {}

    for u, v, r, attrs in G.edges(keys=True, data=True):
        u_type = G.nodes[u].get('type') or G.nodes[u].get('label')
        v_type = G.nodes[v].get('type') or G.nodes[v].get('label')
        if not isinstance(r, str):
            rel = attrs.get('relation') or attrs.get('rel') or attrs.get('type')
        else:
            rel = r
        
        # Replace double underscores with single ones to satisfy PyG requirements
        safe_rel = str(rel).replace('__', '_')
        
        edge_type = (u_type, safe_rel, v_type)
        edge_type = (u_type, safe_rel, v_type)
        
        # Categorize based on node types
        if u_type == 'Patient' or v_type == 'Patient':
            target_dict = dynamic_edges
        else:
            target_dict = static_edges
            
        if edge_type not in target_dict:
            target_dict[edge_type] = []
        
        u_idx = node_mappings[u_type][u]
        v_idx = node_mappings[v_type][v]
        target_dict[edge_type].append([u_idx, v_idx])

    # Finalize Edges in HeteroData
    for etype, content in {**static_edges, **dynamic_edges}.items():
        data[etype].edge_index = torch.tensor(content, dtype=torch.long).t().contiguous()
    
    # 5. Attach the dict to the data object for easy access
    data.static_edge_types = list(static_edges.keys())
    data.dynamic_edge_types = list(dynamic_edges.keys())
    
    print(f"HeteroData created: {len(data.node_types)} node types, {len(data.static_edge_types) + len(data.dynamic_edge_types)} edge types.")
    return data, node_mappings

In [82]:
data, node_mappings = convert_to_hetero_data(G)
data

Starting conversion from NetworkX to HeteroData...
Computing topological features...
HeteroData created: 25 node types, 871 edge types.


HeteroData(
  static_edge_types=[862],
  dynamic_edge_types=[9],
  Patient={
    y=[691],
    x=[691, 4444],
    topo_features=[691, 5],
    topo_scaler_mean=[5],
    topo_scaler_scale=[5],
    num_nodes=691,
  },
  Gene={ num_nodes=808 },
  Abundance={ num_nodes=740 },
  BiologicalProcess={ num_nodes=811 },
  Activity={ num_nodes=763 },
  Pathology={ num_nodes=126 },
  MicroRna={ num_nodes=46 },
  Protein={ num_nodes=2245 },
  Rna={ num_nodes=768 },
  Translocation={ num_nodes=171 },
  Reaction={ num_nodes=17 },
  Degradation={ num_nodes=79 },
  CellSecretion={ num_nodes=72 },
  CellSurfaceExpression={ num_nodes=25 },
  Complex={ num_nodes=672 },
  Composite={ num_nodes=210 },
  Products={ num_nodes=4 },
  Reactants={ num_nodes=4 },
  FusionProtein={ num_nodes=1 },
  Pmod={ num_nodes=34 },
  Fragment={ num_nodes=91 },
  Variant={ num_nodes=58 },
  Location={ num_nodes=28 },
  FromLocation={ num_nodes=21 },
  ToLocation={ num_nodes=26 },
  (Gene, decreases, Protein)={ edge_index=[2, 6]

In [84]:
data['Patient'].topo_features.size(1)

5

In [69]:
import torch
from torch_geometric.data import HeteroData

def merge_patient_protein_edges(data):
    """Rebuild HeteroData() with merged Patient-Protein edges and keep all metadata."""
    new_data = HeteroData()
    
    # 1. Copy global graph-level attributes (e.g., static_edge_types, dynamic_edge_types)
    for key, value in data.to_dict().items():
        if not isinstance(value, dict) and key not in ['static_edge_types', 'dynamic_edge_types']:
            new_data[key] = value
    # 2. Copy node features and properties
    for node_type in data.node_types:
        for key, value in data[node_type].items():
            new_data[node_type][key] = value

    # Collect edges
    d_edges = []
    rev_d_edges = []

    h_edges = []
    rev_h_edges = []

    # 3. Process edge types
    for et in data.edge_types:
        src, rel, dst = et
        edge_index = data[et].edge_index

        # Patient <-> Protein relations to merge
        if (src == 'Patient' and dst == 'Protein') or (src == 'Protein' and dst == 'Patient'):

            # reverse edges
            if 'rev' in rel:
                if 'disease' in rel:
                    rev_d_edges.append(edge_index)
                else:
                    rev_h_edges.append(edge_index)
            # forward edges
            else:
                if 'disease' in rel:
                    d_edges.append(edge_index)
                else:
                    h_edges.append(edge_index)

        # keep other edge types intact
        else:
            new_data[et].edge_index = edge_index

    # 4. Assign the newly merged edge indices
    if d_edges:
        new_data[('Patient', 'reg_disease', 'Protein')].edge_index = torch.cat(d_edges, dim=1)
    if h_edges:
        new_data[('Patient', 'reg_control', 'Protein')].edge_index = torch.cat(h_edges, dim=1)
    if rev_d_edges:
        new_data[('Protein', 'rev_reg_disease', 'Patient')].edge_index = torch.cat(rev_d_edges, dim=1)
    if rev_h_edges:
        new_data[('Protein', 'rev_reg_control', 'Patient')].edge_index = torch.cat(rev_h_edges, dim=1)

    static_edges_list = []
    dynamic_edges_list = []
    for et in new_data.edge_types:
        src, rel, dst = et
        if src == 'Patient' or dst == 'Patient':
            dynamic_edges_list.append(et)
        else:
            static_edges_list.append(et)

    new_data.static_edge_types = static_edges_list
    new_data.dynamic_edge_types = dynamic_edges_list

    return new_data

In [70]:
new_data = merge_patient_protein_edges(data)
new_data

HeteroData(
  static_edge_types=[862],
  dynamic_edge_types=[5],
  Patient={
    y=[691],
    x=[691, 4444],
    topo_features=[1],
    topo_scaler_mean=[1],
    topo_scaler_scale=[5],
    num_nodes=691,
  },
  Gene={ num_nodes=808 },
  Abundance={ num_nodes=740 },
  BiologicalProcess={ num_nodes=811 },
  Activity={ num_nodes=763 },
  Pathology={ num_nodes=126 },
  MicroRna={ num_nodes=46 },
  Protein={ num_nodes=2245 },
  Rna={ num_nodes=768 },
  Translocation={ num_nodes=171 },
  Reaction={ num_nodes=17 },
  Degradation={ num_nodes=79 },
  CellSecretion={ num_nodes=72 },
  CellSurfaceExpression={ num_nodes=25 },
  Complex={ num_nodes=672 },
  Composite={ num_nodes=210 },
  Products={ num_nodes=4 },
  Reactants={ num_nodes=4 },
  FusionProtein={ num_nodes=1 },
  Pmod={ num_nodes=34 },
  Fragment={ num_nodes=91 },
  Variant={ num_nodes=58 },
  Location={ num_nodes=28 },
  FromLocation={ num_nodes=21 },
  ToLocation={ num_nodes=26 },
  (Gene, decreases, Protein)={ edge_index=[2, 6] },
 

In [72]:
# Feature construction
def build_data_dict(data):
    """
    Construct x_dict:
    - Patient: use real features
    - Others: zero vectors
    """
    x_dict = {}

    for node_type in data.node_types:
        try:
            x_dict[node_type] = data[node_type]['x']
        except KeyError:
            x_dict[node_type]=None
        
    data.x_dict = x_dict
    data.num_nodes_dict = {ntype: data[ntype].num_nodes for ntype in data.node_types}
    data.static_edge_index_dict = {etype: data[etype].edge_index for etype in data.static_edge_types}
    data.dynamic_edge_index_dict = {etype: data[etype].edge_index for etype in data.dynamic_edge_types}
    data.edge_index_dict = {**data.static_edge_index_dict, **data.dynamic_edge_index_dict}
    
    return data

In [73]:
n_data = build_data_dict(new_data)
n_data

HeteroData(
  static_edge_types=[862],
  dynamic_edge_types=[5],
  x_dict={
    Patient=[691, 4444],
    Gene=None,
    Abundance=None,
    BiologicalProcess=None,
    Activity=None,
    Pathology=None,
    MicroRna=None,
    Protein=None,
    Rna=None,
    Translocation=None,
    Reaction=None,
    Degradation=None,
    CellSecretion=None,
    CellSurfaceExpression=None,
    Complex=None,
    Composite=None,
    Products=None,
    Reactants=None,
    FusionProtein=None,
    Pmod=None,
    Fragment=None,
    Variant=None,
    Location=None,
    FromLocation=None,
    ToLocation=None,
  },
  num_nodes_dict={
    Patient=691,
    Gene=808,
    Abundance=740,
    BiologicalProcess=811,
    Activity=763,
    Pathology=126,
    MicroRna=46,
    Protein=2245,
    Rna=768,
    Translocation=171,
    Reaction=17,
    Degradation=79,
    CellSecretion=72,
    CellSurfaceExpression=25,
    Complex=672,
    Composite=210,
    Products=4,
    Reactants=4,
    FusionProtein=1,
    Pmod=34,
    Frag

In [80]:
type(n_data['Patient'].topo_features)

tuple

In [85]:
path = "../../GateEmbeddingTask/HRGNN/results/ANMerge/AD_KG/dual_hybrid/ecdf_5/hrgat/results.pkl"
with open(path, 'rb') as f:
    results = pickle.load(f)
results

{'Fuzzyhrgat_all_rules': {'fold_test_metrics': [{'accuracy': 0.6406926406926406,
    'balanced_accuracy': 0.5,
    'f1': 0.7810026385224275,
    'f1_macro': 0.39050131926121373,
    'f1_weighted': 0.5003826428628539,
    'auroc': 0.5661836535330511,
    'auprc': 0.7207873412654178},
   {'accuracy': 0.6391304347826087,
    'balanced_accuracy': 0.5,
    'f1': 0.7798408488063661,
    'f1_macro': 0.38992042440318303,
    'f1_weighted': 0.49842002075885133,
    'auroc': 0.658306696172445,
    'auprc': 0.7931947464413149},
   {'accuracy': 0.6391304347826087,
    'balanced_accuracy': 0.5,
    'f1': 0.7798408488063661,
    'f1_macro': 0.38992042440318303,
    'f1_weighted': 0.49842002075885133,
    'auroc': 0.5874928284566838,
    'auprc': 0.7448571335184762}],
  'fuzzy_rules': [tensor([[0.2771, 0.5058, 0.4473,  ..., 0.1922, 0.1922, 0.3794],
           [0.3283, 0.5879, 0.5865,  ..., 0.2744, 0.2743, 0.3761],
           [0.2691, 0.7643, 0.6431,  ..., 0.3763, 0.3763, 0.3067],
           ...,
    

In [87]:
import pandas as pd
long_format_rows = []
for model_name, res in results.items():
    res_metrics = res['fold_test_metrics']
    for fold_idx, metrics in enumerate(res_metrics):
        long_format_rows.append({'model':model_name,
                                'fold_idx':fold_idx,
                                **metrics})
df = pd.DataFrame(long_format_rows)
df

,model,fold_idx,accuracy,balanced_accuracy,f1,f1_macro,f1_weighted,auroc,auprc
0,Fuzzyhrgat_all_rules,0,0.640693,0.5,0.781003,0.390501,0.500383,0.566184,0.720787
1,Fuzzyhrgat_all_rules,1,0.639130,0.5,0.779841,0.389920,0.498420,0.658307,0.793195
2,Fuzzyhrgat_all_rules,2,0.639130,0.5,0.779841,0.389920,0.498420,0.587493,0.744857
